In [ ]:
import tempfile
import os
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

import oq_wrapper as oqw
import nn_gmm as nng
import ml_tools as mlt

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    level=logging.WARNING
)

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]


In [ ]:
result_dir = "/Users/claudy/dev/work/data/nn_gmm/results/0623_0935_10Epochs_simpleFeatures"

In [ ]:
# Injected Parameters
result_dir = "/Users/claudy/dev/work/data/nn_gmm/results/0623_0935_10Epochs_simpleFeatures"


In [ ]:
result_dir = Path(result_dir)
empdb_ffp = Path(os.environ["wdata"]) / "nn_gmm/20250616_CS200m_empdb.db"

In [ ]:
display(Markdown("## Inputs\n\n"))
display(Markdown(f"**Result directory**: `{result_dir}`\n\n**Empirical database**: `{empdb_ffp}`"))

In [ ]:
run_config = nng.RunConfig.from_yaml(result_dir / "run_config.yaml")
val_pred_df = pd.read_parquet(result_dir / "val_results.parquet").sort_index()
metrics = pd.read_parquet(result_dir / "metrics.parquet")

In [ ]:
val_record_int_ids = val_pred_df.index.values.astype(int)

with nng.IMDB(run_config.imdb_ffp, readonly=True) as imdb:
    event_df = imdb.get_event_df()
    site_df = imdb.get_site_df()
    val_sim_df = imdb.get_im_data_tmp_table(run_config.ims, val_record_int_ids).sort_index()
    val_record_info_df = imdb.get_record_info_df(record_int_ids=val_record_int_ids)
    val_site_event_df = imdb.get_site_event_df()

val_sim_df["event_int_id"] = val_record_info_df.loc[val_sim_df.index, "event_int_id"].values
val_sim_df["site_int_id"] = val_record_info_df.loc[val_sim_df.index, "site_int_id"].values
val_sim_df["site_event_int_id"] = nng.utils.get_site_event_int_id(val_sim_df.site_int_id, val_sim_df.event_int_id)
val_sim_df["magnitude"] = event_df.loc[val_sim_df["event_int_id"], "magnitude"].values
val_sim_df["rrup"] = val_site_event_df.loc[val_sim_df["site_event_int_id"], "rrup"].values

val_pred_df["site_int_id"] = val_record_info_df.loc[val_pred_df.index, "site_int_id"].values

assert val_pred_df.index.equals(val_sim_df.index)

In [ ]:
with nng.EmpiricalDB(empdb_ffp, readonly=True) as empdb:
    val_emp_df = empdb.get_gm_params_tmp_table(val_pred_df.index.values.astype(int))

assert val_emp_df.index.equals(val_pred_df.index)
assert val_emp_df.index.equals(val_sim_df.index)

In [ ]:
res_df = pd.DataFrame(data=np.log(val_sim_df[run_config.ims].values) - val_pred_df[run_config.pred_mean_keys].values, index=val_pred_df.index, columns=run_config.ims)
emp_res_df = pd.DataFrame(data=np.log(val_sim_df[nng.constants.PSA_KEYS].values) - val_emp_df[nng.constants.GMM_PSA_MEAN_KEYS].values, index=val_emp_df.index, columns=nng.constants.PSA_KEYS)

## Metrics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

mlt.plotting.plot_metrics(history=metrics.to_dict("list"), metric_keys=["loss_hist"], ax=ax1, y_label="Loss")
mlt.plotting.plot_metrics(history=metrics.to_dict("list"), metric_keys=["mse_hist"], ax=ax2, y_label="MSE")

fig.tight_layout()

## Model Bias & Residual Standard Deviation

In [ ]:
model_bias, res_std = res_df.mean(axis=0), res_df.std(axis=0)
emp_model_bias, emp_res_std = emp_res_df.mean(axis=0), emp_res_df.std(axis=0)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.8, 0.8), std_y_axis_limits=(0.0, 0.75))

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="b")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="g")

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="b")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="g")

if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

### Magnitude

In [ ]:
mag_bins = [5.5, 6.5, 7.5, 8.5]
mag_bin_labels = [f"$M_w$: {mag_bins[i]} - {mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]
mag_bin_colors = ["blue", "purple", "red"]

In [ ]:
val_sim_df["mag_bin"] = pd.cut(val_sim_df["magnitude"], bins=mag_bins, labels=mag_bin_labels, include_lowest=True)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.8, 0.8), std_y_axis_limits=(0.0, 0.75))

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="k")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_mag_bin in enumerate(mag_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["mag_bin"] == cur_mag_bin].index.values
    cur_model_bias = res_df.loc[cur_record_ids].mean(axis=0)
    cur_emp_model_bias = emp_res_df.loc[cur_record_ids].mean(axis=0)
    ax1.plot(nng.constants.PSA_PERIODS, cur_emp_model_bias[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], linestyle="--")
    ax1.plot(nng.constants.PSA_PERIODS, cur_model_bias[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], label=f"{cur_mag_bin}, N={len(cur_record_ids)}")

ax1.legend()

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="k")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_mag_bin in enumerate(mag_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["mag_bin"] == cur_mag_bin].index.values
    cur_res_std = res_df.loc[cur_record_ids].std(axis=0)
    cur_emp_res_std = emp_res_df.loc[cur_record_ids].std(axis=0)
    ax3.plot(nng.constants.PSA_PERIODS, cur_emp_res_std[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], linestyle="--")
    ax3.plot(nng.constants.PSA_PERIODS, cur_res_std[nng.constants.PSA_KEYS].values, c=mag_bin_colors[i], label=cur_mag_bin)


if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

### $R_{Rup}$

In [ ]:
rrup_bins = [0, 30, 100, 500]
rrup_bin_labels = ["$R_{Rup}$ 0 - 30", "$R_{Rup}$ 30 - 100", "$R_{Rup}$ 100 - 500"]
rrup_bin_colors = ["blue", "purple", "red"]

In [ ]:
val_sim_df["rrup_bin"] = pd.cut(val_sim_df["rrup"], bins=rrup_bins, labels=rrup_bin_labels, include_lowest=True)

In [ ]:
fig, ax1, ax2, ax3, ax4 = nng.plot_utils.get_bias_residual_fig(figsize=(16, 6), bias_y_axis_limits=(-0.8, 0.8), std_y_axis_limits=(0.0, 0.75))

ax1.plot(nng.constants.PSA_PERIODS, model_bias[nng.constants.PSA_KEYS].values, c="k")
ax1.plot(nng.constants.PSA_PERIODS, emp_model_bias[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_rrup_bin in enumerate(rrup_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["rrup_bin"] == cur_rrup_bin].index.values
    cur_model_bias = res_df.loc[cur_record_ids].mean(axis=0)
    cur_emp_model_bias = emp_res_df.loc[cur_record_ids].mean(axis=0)
    ax1.plot(nng.constants.PSA_PERIODS, cur_emp_model_bias[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], linestyle="--")
    ax1.plot(nng.constants.PSA_PERIODS, cur_model_bias[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], label=f"{cur_rrup_bin} (km), N={len(cur_record_ids)}")

ax1.legend()

if run_config.im_set == "all":
    ax2.scatter(nng.constants.NON_PSA_IMS, model_bias[nng.constants.NON_PSA_IMS].values, c="b")

ax3.plot(nng.constants.PSA_PERIODS, res_std[nng.constants.PSA_KEYS].values, c="k")
ax3.plot(nng.constants.PSA_PERIODS, emp_res_std[nng.constants.PSA_KEYS].values, c="k", linestyle="--")

for i, cur_rrup_bin in enumerate(rrup_bin_labels):
    cur_record_ids = val_sim_df.loc[val_sim_df["rrup_bin"] == cur_rrup_bin].index.values
    cur_res_std = res_df.loc[cur_record_ids].std(axis=0)
    cur_emp_res_std = emp_res_df.loc[cur_record_ids].std(axis=0)
    ax3.plot(nng.constants.PSA_PERIODS, cur_emp_res_std[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], linestyle="--")
    ax3.plot(nng.constants.PSA_PERIODS, cur_res_std[nng.constants.PSA_KEYS].values, c=rrup_bin_colors[i], label=cur_rrup_bin)


if run_config.im_set == "all":
    ax4.scatter(nng.constants.NON_PSA_IMS, res_std[nng.constants.NON_PSA_IMS].values, c="b")

## Model Trends

### Magnitude

In [ ]:
# Define fixed inputs and limits for validation data points
fixed_inputs = dict(vs30=500,
    z1p0=0.13,
    z2p5=1,
    tect_type=nng.constants.REVERSE_TECTONIC_TYPE_MAPPING[oqw.constants.TectType.ACTIVE_SHALLOW],
    rake=10,
    dip=60,
    dtop=0,
    dbottom=15,
    hypo_depth=8.5
)
mag_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "rrup": (5, 5),
}

#### Short Distance ($R_{Rup} = 25$ km)

In [ ]:
short_dist_fixed_inputs = {"rrup": 25, "rjb": 25, "rx": 25} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, short_dist_fixed_inputs, mag_limits, val_record_int_ids, device);

#### Moderate Distance ($R_{Rup} = 75$ km)

In [ ]:
moderate_dist_fixed_inputs = {"rrup": 75, "rjb": 75, "rx": 75} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, moderate_dist_fixed_inputs, mag_limits, val_record_int_ids, device);

#### Long Distance ($R_{Rup} = 125$ km)

In [ ]:
long_dist_fixed_inputs = {"rrup": 125, "rjb": 125, "rx": 125} | fixed_inputs

nng.plots.magnitude_trend_plot(result_dir, val_sim_df, long_dist_fixed_inputs, mag_limits, val_record_int_ids, device);

### Source-To-Site Distance

In [ ]:
rrup_limits = {
        "vs30": (25, 25),
        "z1p0": (0.25, 0.25),
        "magnitude": (0.1, 0.1),
}

In [ ]:
moderate_mag_fixed_inputs = {"magnitude": 6.6} | fixed_inputs

nng.plots.rrup_trend_plot(result_dir, val_sim_df, moderate_mag_fixed_inputs, rrup_limits, val_record_int_ids, device);

## Spatial

### Bias & Residual Standard Deviation

In [ ]:
figs = nng.plots_spatial.site_bias_res_std(val_pred_df, val_sim_df, site_df, PLOT_IMS, run_config)

In [ ]:
for cur_im in PLOT_IMS:
    fig = nng.plot_utils.pygmt_side_by_side(figs[cur_im][0], figs[cur_im][1], title_1=f"{nng.utils.get_nice_im_name(cur_im)} - Bias", title_2=f"{nng.utils.get_nice_im_name(cur_im)} - Residual Standard Deviation")
    plt.show()